In [ ]:

from pyspark.sql import functions as F

# Separate synthetic input for testing data quality failures.
# This is NOT the real 2025 flight dataset.

test_data = [
    # Four valid flights
    (2025, 1, 10, "2025-01-10", "AA", "JFK", "LAX", 800, 0, 0, 20.0, 1),
    (2025, 1, 11, "2025-01-11", "DL", "ATL", "ORD", 900, 0, 0, 5.0, 0),
    (2025, 2, 12, "2025-02-12", "UA", "SFO", "DEN", 1500, 0, 0, 35.0, 1),
    (2025, 3, 13, "2025-03-13", "WN", "DAL", "LAS", 1800, 0, 0, -5.0, 0),

    # Six invalid flights
    (2024, 1, 10, "2024-01-10", "AA", "JFK", "LAX", 800, 0, 0, 20.0, 1),
    (2025, 2, 12, "2025-02-12", "DL", None, "ORD", 900, 0, 0, 5.0, 0),
    (2025, 2, 13, "2025-02-13", "UA", "SFO", "DEN", 1500, 1, 0, None, 0),
    (2025, 3, 14, "2025-03-14", "WN", "DAL", "LAS", 1800, 0, 1, None, 0),
    (2025, 4, 15, "2025-04-15", "AA", "MIA", "JFK", 2560, 0, 0, 20.0, 1),
    (2025, 5, 16, "2025-05-16", "DL", "ATL", "LAX", 1100, 0, 0, 30.0, 0)
]

columns = [
    "Year", "Month", "DayofMonth", "FlightDate",
    "Reporting_Airline", "Origin", "Dest",
    "CRSDepTime", "Cancelled", "Diverted",
    "ArrDelay", "ArrDel15"
]

test_df = spark.createDataFrame(test_data, columns)

print("Synthetic test records:", test_df.count())
display(test_df)


In [ ]:

# Match the validation logic from the production ETL notebook.

checked = (
    test_df
    .withColumn("YearNum", F.col("Year").cast("int"))
    .withColumn("MonthNum", F.col("Month").cast("int"))
    .withColumn("FlightDateParsed", F.to_date("FlightDate"))
    .withColumn("CancelledNum", F.col("Cancelled").cast("double"))
    .withColumn("DivertedNum", F.col("Diverted").cast("double"))
    .withColumn("ArrDelayNum", F.col("ArrDelay").cast("double"))
    .withColumn("TargetNum", F.col("ArrDel15").cast("double"))
    .withColumn("CRSDepNum", F.col("CRSDepTime").cast("int"))
)

valid_time = (
    (
        F.col("CRSDepNum").between(0, 2359)
        & ((F.col("CRSDepNum") % 100) < 60)
    )
    | (F.col("CRSDepNum") == 2400)
)

valid = (
    (F.col("YearNum") == 2025)
    & F.col("MonthNum").between(1, 12)
    & F.col("FlightDateParsed").isNotNull()
    & (F.year("FlightDateParsed") == F.col("YearNum"))
    & (F.month("FlightDateParsed") == F.col("MonthNum"))
    & (F.dayofmonth("FlightDateParsed") == F.col("DayofMonth").cast("int"))
    & F.col("Reporting_Airline").isNotNull()
    & (F.trim("Reporting_Airline") != "")
    & F.col("Origin").isNotNull()
    & (F.trim("Origin") != "")
    & F.col("Dest").isNotNull()
    & (F.trim("Dest") != "")
    & valid_time
    & (F.col("CancelledNum") == 0)
    & (F.col("DivertedNum") == 0)
    & F.col("ArrDelayNum").isNotNull()
    & F.col("TargetNum").isin(0.0, 1.0)
    & (
        F.col("TargetNum")
        == F.when(F.col("ArrDelayNum") >= 15, 1.0).otherwise(0.0)
    )
)

checked = checked.withColumn(
    "QualityStatus",
    F.when(valid, "ACCEPTED").otherwise("REJECTED")
)

display(checked.select(
    "FlightDate", "Reporting_Airline", "Origin",
    "Cancelled", "Diverted", "ArrDelay",
    "ArrDel15", "QualityStatus"
))

counts = {
    r["QualityStatus"]: r["count"]
    for r in checked.groupBy("QualityStatus").count().collect()
}

total = sum(counts.values())
accepted = counts.get("ACCEPTED", 0)
rejected = counts.get("REJECTED", 0)

print("Total test records:", total)
print("Accepted:", accepted)
print("Rejected:", rejected)
print(f"Acceptance rate: {accepted / total:.2%}")

assert total == 10
assert accepted == 4
assert rejected == 6

print("PASS: Invalid records correctly detected")


In [ ]:

# Controlled failure test — no Azure data is written.

def enforce_quality_gate(total_rows, accepted_rows, minimum_rate=0.90):
    if total_rows <= 0:
        raise ValueError("QUALITY FAILURE: Empty dataset")

    rate = accepted_rows / total_rows

    if rate < minimum_rate:
        raise ValueError(
            f"QUALITY FAILURE: {rate:.2%} is below "
            f"the {minimum_rate:.0%} threshold. "
            "Publication blocked."
        )

    return True

failure_detected = False

try:
    enforce_quality_gate(total, accepted)
except ValueError as error:
    failure_detected = True
    print("EXPECTED FAILURE:", error)

assert failure_detected, "ERROR: Quality gate did not reject invalid input"

print("PASS: Quality gate rejected the bad input")
print("PASS: No data was published")
